# Hyperparam Tuning for the MNIST MLP

In [ ]:
from ast import literal_eval
from copy import deepcopy

import graphviz
import matrepr
import torch
import torchinfo
from safetensors.torch import load_model, save_model
from torch import nn
from torch.nn import (
    CrossEntropyLoss,
    KLDivLoss,
    L1Loss,
    MSELoss,
    CosineSimilarity,
    HuberLoss,
)
from torch.nn import functional as F
from torch.optim import SGD, Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from learningdynamics.models import MLP, init_weights
from learningdynamics.utils import (
    compute_model_accuracy,
    getActivation,
    safetensors_metadata_parser,
)
import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from functools import partial

In [ ]:
# | hide
matrepr.params.title = True
matrepr.params.indices = False
matrepr.params.precision = 4
matrepr.cell_align = "left"
graphviz.set_jupyter_format("png")

%matplotlib inline

In [ ]:
file_path = "/workspaces/learningdynamics/saved_weights/mnist_model.safetensors"

# Parse metadata
metadata = safetensors_metadata_parser(file_path=file_path)

# Load model
model = MLP(
    config=literal_eval(metadata["config"]),
    nonlinearity=metadata["nonlinearity"],
    in_features=784,
    out_features=10,
)
_ = load_model(model, file_path, device="cuda:0")

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),  # first, convert image to PyTorch tensor
        transforms.Normalize((0.1307,), (0.3081,)),  # normalize inputs
    ]
)
train_set = datasets.MNIST(
    root="/mnt/datasets/", train=True, download=True, transform=transform
)
test_set = datasets.MNIST(
    root="/mnt/datasets/", train=False, download=True, transform=transform
)

In [ ]:
print(f"Testing Accuracy: {compute_model_accuracy(model, test_set)}")

Testing Accuracy: 0.972000002861023


In [ ]:
# torchinfo.summary(model, row_settings=["var_names", "ascii_only"])
model

MLP(
  (features): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [ ]:
def freeze_and_scale_model(
    model,
    start,
    layers_to_add=1,
    device="cuda:0",
):
    # Set up scaled model
    model.to(device)
    scaled_model = deepcopy(model)

    # Update model
    scaled_container = torch.nn.Sequential()
    for idx, module in enumerate(scaled_model.features):
        # Copy over weights and biases
        if hasattr(module, "weight"):
            module.weight.requires_grad_(False)
        if hasattr(module, "bias"):
            module.bias.requires_grad_(False)
        scaled_container.append(module)
        # Add new layers
        if idx == start:
            prev_out = model.features[idx - 1].out_features
            for _ in range(layers_to_add):
                linear_layer = nn.Linear(prev_out, prev_out)
                linear_layer.apply(init_weights)
                scaled_container.append(linear_layer)
                scaled_container.append(nn.ReLU())
    scaled_model.features = scaled_container
    scaled_model.to(device=device)

    scaled_model.to(device="cpu")
    return scaled_model

In [ ]:
DEVICE = "cuda:0"
PRINT_FREQ = 1

NUM_LAYERS_ADDED = 10
LAYER_START = 5
DIMENSIONALITY = model.features[LAYER_START + 1].in_features
LAYER_END = LAYER_START + 2

BATCH_SIZE = 4_096
NUM_EPOCHS = 30

dataloader = DataLoader(dataset=train_set, shuffle=True, batch_size=BATCH_SIZE)

In [ ]:
scaled_model = freeze_and_scale_model(
    model,
    start=LAYER_START,
    layers_to_add=NUM_LAYERS_ADDED,
)

In [ ]:
torchinfo.summary(scaled_model, row_settings=["var_names", "ascii_only"])
# scaled_model

Layer (type (var_name))                  Param #
MLP (MLP)                                --
+ Sequential (features)                  --
|    + Linear (0)                        (200,960)
|    + ReLU (1)                          --
|    + Linear (2)                        (32,896)
|    + ReLU (3)                          --
|    + Linear (4)                        (8,256)
|    + ReLU (5)                          --
|    + Linear (6)                        4,160
|    + ReLU (7)                          --
|    + Linear (8)                        4,160
|    + ReLU (9)                          --
|    + Linear (10)                       4,160
|    + ReLU (11)                         --
|    + Linear (12)                       4,160
|    + ReLU (13)                         --
|    + Linear (14)                       4,160
|    + ReLU (15)                         --
|    + Linear (16)                       4,160
|    + ReLU (17)                         --
|    + Linear (18)                 

In [ ]:
print(f"Testing Accuracy: {compute_model_accuracy(scaled_model.to('cpu'), test_set)}")

Testing Accuracy: 0.12039999663829803


In [ ]:
# Define the training function
def train_model(config):

    model = MLP(
        config=literal_eval(metadata["config"]),
        nonlinearity=metadata["nonlinearity"],
        in_features=784,
        out_features=10,
        )
    
    _ = load_model(model, file_path, device="cuda:0")

    scaled_model = freeze_and_scale_model(
        model,
        start=LAYER_START,
        layers_to_add=NUM_LAYERS_ADDED,
    )
    scaled_model.to(DEVICE)
    scaled_model.train()

    transform = transforms.Compose(
        [
            transforms.ToTensor(),  # first, convert image to PyTorch tensor
            transforms.Normalize((0.1307,), (0.3081,)),  # normalize inputs
        ]
    )
    train_set = datasets.MNIST(
        root="/mnt/datasets/", train=True, download=True, transform=transform
    )

    dataloader = DataLoader(dataset=train_set, shuffle=True, batch_size=BATCH_SIZE)
    activation_dict = {}

    # Setup training
    huber_criterion = HuberLoss(delta=1)
    optimizer = AdamW(
        scaled_model.parameters(),
        lr=config["lr"],
        betas=(config["beta1"], config["beta2"]),
    )

    hooks = []
    hooks.append(
        model.features[LAYER_END].register_forward_hook(
            getActivation(f"layer_{LAYER_END}", activation_dict)
        )
    )
    new_end = LAYER_END + NUM_LAYERS_ADDED * 2
    hooks.append(
        scaled_model.features[new_end].register_forward_hook(
            getActivation(f"scaled_layer_{new_end}", activation_dict, detach=False)
        )
    )

    loss_vector = []
    for epoch in range(NUM_EPOCHS):
        running_loss = 0.0

        for input, label in dataloader:
            input, label = input.to(DEVICE), torch.flatten(label.to(DEVICE))
            output = scaled_model(input)

            with torch.no_grad():
                _ = model(input)

            optimizer.zero_grad()
            output_activation = activation_dict[f"scaled_layer_{new_end}"].to(DEVICE)
            target_activation = activation_dict[f"layer_{LAYER_END}"].to(DEVICE)
            huber_loss = huber_criterion(output_activation, target_activation)

            loss = huber_loss
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(dataloader)
        loss_vector.append(epoch_loss)

        ray.train.report(dict(loss=epoch_loss))

    for hook in hooks:
        hook.remove()

In [ ]:

# Define the search space
search_space = {
    "lr": tune.qloguniform(1e-3, 3e-3, 1e-3),
    "beta1": tune.qloguniform(0.6, 0.9, 1e-1),
    "beta2": tune.qloguniform(0.7, 0.99, 1e-2),
}

# Configure the scheduler
asha_scheduler = ASHAScheduler(
    metric="loss",
    mode="min",
    max_t=NUM_EPOCHS,
    grace_period=NUM_EPOCHS//2,
    reduction_factor=4
)

# Run the hyperparameter tuning
analysis = tune.run(
    partial(train_model),
    config=search_space,
    num_samples=50,  # Number of different hyperparameter combinations to try
    scheduler=asha_scheduler,
    resources_per_trial={"cpu": 4, "gpu": 0.05}  # Adjust based on your resources
)
print("Best hyperparameters found were: ", analysis.get_best_config("loss", mode="min"))


2024-08-12 10:25:41,687	INFO worker.py:1781 -- Started a local Ray instance.
2024-08-12 10:25:42,254	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `tune.run(...)`.
2024-08-12 10:25:42,255	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


Trial name,loss
train_model_3a437_00000,0.539696
train_model_3a437_00001,0.513961
train_model_3a437_00002,0.614432
train_model_3a437_00003,0.375632
train_model_3a437_00004,0.669422
train_model_3a437_00005,0.289516
train_model_3a437_00006,0.495417
train_model_3a437_00007,0.604721
train_model_3a437_00008,0.382948
train_model_3a437_00009,0.547773


2024-08-12 10:38:02,487	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/train_model_2024-08-12_10-25-42' in 0.0340s.
2024-08-12 10:38:02,502	INFO tune.py:1041 -- Total run time: 740.25 seconds (740.02 seconds for the tuning loop).


Best hyperparameters found were:  {'lr': 0.003, 'beta1': 0.9, 'beta2': 0.71}


In [ ]:
df = analysis.results_df
sorted_df = df.sort_values(by="loss", ascending=True)
display(sorted_df.head(20))

,loss,timestamp,checkpoint_dir_name,done,training_iteration,date,time_this_iter_s,time_total_s,pid,hostname,node_ip,time_since_restore,iterations_since_restore,experiment_tag,config/lr,config/beta1,config/beta2
trial_id,,,,,,,,,,,,,,,,,
3a437_00024,0.285159,1723458776,None,True,30,2024-08-12_10-32-56,6.538230,192.753435,990830,e1bb1df5c343,172.17.0.3,192.753435,30,"24_beta1=0.9000,beta2=0.7100,lr=0.0030",0.003,0.9,0.71
3a437_00005,0.289516,1723458580,None,True,30,2024-08-12_10-29-40,6.560401,235.101765,987450,e1bb1df5c343,172.17.0.3,235.101765,30,"5_beta1=0.9000,beta2=0.9700,lr=0.0030",0.003,0.9,0.97
3a437_00010,0.293950,1723458578,None,True,30,2024-08-12_10-29-38,6.588308,232.899250,987464,e1bb1df5c343,172.17.0.3,232.899250,30,"10_beta1=0.9000,beta2=0.9600,lr=0.0010",0.001,0.9,0.96
3a437_00027,0.301041,1723458779,None,True,30,2024-08-12_10-32-59,6.470589,191.459625,991244,e1bb1df5c343,172.17.0.3,191.459625,30,"27_beta1=0.9000,beta2=0.9700,lr=0.0010",0.001,0.9,0.97
3a437_00038,0.302911,1723458880,None,True,30,2024-08-12_10-34-40,6.381341,191.334997,993002,e1bb1df5c343,172.17.0.3,191.334997,30,"38_beta1=0.8000,beta2=0.9700,lr=0.0010",0.001,0.8,0.97
3a437_00039,0.313689,1723458968,None,True,30,2024-08-12_10-36-08,6.328969,191.934958,993409,e1bb1df5c343,172.17.0.3,191.934958,30,"39_beta1=0.8000,beta2=0.9000,lr=0.0010",0.001,0.8,0.90
3a437_00034,0.313798,1723458873,None,True,30,2024-08-12_10-34-33,6.466693,191.139796,992379,e1bb1df5c343,172.17.0.3,191.139796,30,"34_beta1=0.9000,beta2=0.8000,lr=0.0020",0.002,0.9,0.80
3a437_00048,0.344847,1723459082,None,True,30,2024-08-12_10-38-02,6.350430,203.740959,995002,e1bb1df5c343,172.17.0.3,203.740959,30,"48_beta1=0.9000,beta2=0.7700,lr=0.0010",0.001,0.9,0.77
3a437_00045,0.346764,1723458976,None,True,30,2024-08-12_10-36-16,6.465058,191.142362,994226,e1bb1df5c343,172.17.0.3,191.142362,30,"45_beta1=0.9000,beta2=0.9700,lr=0.0010",0.001,0.9,0.97


In [ ]:
len(df)

50